# Sandbox - Quick Experimentation

Use this notebook to quickly test ideas, compare speaker variants, etc.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from rsa import *
from rsa.experimental import *

## Quick Setup

In [ ]:
# One-liner agent setup
agents = make_agents(theta=0.3, n=1, m=7, alpha=3.0, speaker_psi='high', listener_type='vig')
world, semantics = agents['world'], agents['semantics']
thetas = agents['thetas']
s0, l0, s1, l1 = agents['s0'], agents['l0'], agents['s1'], agents['l1']

## Compare Speaker Variants

In [ ]:
alpha = 3.0
obs = (0, 0, 0, 0, 0, 0, 0, 1)  # 0/7 effective

# Paper version (obs informativeness + default persuasiveness)
s1_paper = Speaker1(thetas, l0, semantics=semantics, world=world, alpha=alpha, psi='high')

# Variant: state-level informativeness + default persuasiveness
s1_state = Speaker1_state_inf_def_pers(thetas, l0, semantics=semantics, world=world, alpha=alpha, psi='high')

# Variant: obs informativeness + min-normalized persuasiveness
s1_new_pers = Speaker1_obs_inf_new_pers1(thetas, l0, semantics=semantics, world=world, alpha=alpha, psi='high')

# Variant: state-level + goal distribution persuasiveness
s1_goal = Speaker1_state_inf_new_pers2(thetas, l0, semantics=semantics, world=world, alpha=alpha, psi='high')

# Variant: state-level + log opinion pool persuasiveness
s1_pool = Speaker1_state_inf_new_pers4(thetas, l0, semantics=semantics, world=world, alpha=alpha, psi='high')

# Compare distributions
variants = [
    ('Paper (obs inf + E[θ])', s1_paper.dist_over_utterances_obs(obs, 'high')),
    ('State inf + E[θ]', s1_state.dist_over_utterances_obs(obs, 'high')),
    ('Obs inf + min-norm', s1_new_pers.dist_over_utterances_obs(obs, 'high')),
    ('State inf + goal dist', s1_goal.dist_over_utterances_obs(obs, 'high')),
    ('State inf + log pool', s1_pool.dist_over_utterances_obs(obs, 'high')),
]

plot_speaker_grid([d for _, d in variants], [n for n, _ in variants], alpha=alpha, ncols=3)
plt.show()

## Quick Dyadic Game

In [ ]:
# Run a quick game
world = make_world(theta=0.3, n=1, m=7)
semantics = make_semantics(n=1)
thetas = make_thetas()
psis = ['inf', 'high', 'low']

s1_result, l1_result, l0_result, s0_result = game_s1(
    thetas, psis, semantics, world,
    speaker_type='low', listener_type='vig',
    alpha=3.0, rounds=50, verbose=False
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
plot_theta_learning(l1_result, true_theta=0.3, title='Theta Learning (vig vs pers-)', ax=ax1)
plot_psi_learning(l1_result, title='Speaker Type Detection', ax=ax2)
plt.show()

## Suspicion Analysis

In [ ]:
# Suspicion scores over the game
if l1_result.suspicion:
    plt.figure(figsize=(10, 4))
    plt.plot(l1_result.suspicion, 'b-', linewidth=1.5)
    plt.xlabel('Round')
    plt.ylabel('Suspicion')
    plt.title('Suspicion Score Over Time')
    plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='threshold=0.5')
    plt.legend()
    plt.show()

## Listener Switch Experiment

In [ ]:
# Test Listener1Switch
world = make_world(theta=0.5, n=1, m=7)
semantics = make_semantics(n=1)
s0 = Speaker0(thetas, semantics=semantics, world=world)
l0 = Listener0(thetas, s0, semantics=semantics, world=world)
s1 = Speaker1(thetas, l0, semantics=semantics, world=world, alpha=3.0, psi='low')

l1_switch = Listener1Switch(thetas, psis, s1, world, semantics, threshold=0.5, alpha=3.0)

for _ in range(30):
    obs = world.sample_obs()
    utt = s1.sample_utterance(obs)
    l1_switch.update(utt)
    s1.update(obs)
    l0.update(utt)
    s0.update(obs)

print(f'Switched: {l1_switch.switched}, Switch point: {l1_switch.switch_point}')
print(f'Final E[θ]: {expected_theta(l1_switch.marginal_theta()):.3f}')

## Scratch Space

Your experiments here...